In [6]:
import gc
import glob
from pathlib import Path

import numpy as np
from scipy.stats import pearsonr, fisher_exact
from statsmodels.stats.multitest import multipletests

import pandas as pd
import scanpy as sc

import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [9]:
CSV_PATH = '/mnt/sdb/scz_meta_analysis_processed/dge_signatures/dreamlet_dges/disease_analyses/'

# by manuscript #
dge_3q29 = pd.read_csv(Path(CSV_PATH) / 'dreamlet_purcell_dge_scz_results.csv')
dge_15q13 = pd.read_csv(Path(CSV_PATH) / 'dreamlet_walsh_15q13_dge_scz_results.csv')
dge_nrxn1 = pd.read_csv(Path(CSV_PATH) / 'dreamlet_nrxn1_combined.csv')
dge_22q11 = pd.read_csv(Path(CSV_PATH) / 'dreamlet_22q11_combined.csv')
dge_idiopathic = pd.read_csv(Path(CSV_PATH) / 'dreamlet_idiopathic_combined.csv')

In [19]:
def clean_limma_dge(df):
    out = df.copy()

    out = out.rename(
        columns={
            'z.std':'DGE_STAT',
            'logFC':'DGE_EFFECT',
            'ID':'SYMBOL'
        }
    )

    return out[['SYMBOL','assay','DGE_EFFECT','DGE_STAT','P.Value','adj.P.Val','dataset']]

def clean_dreamlet(df):

    out = df.copy()

    out = out.rename(
        columns={
            "ID":"SYMBOL",
            "estimate":"DGE_EFFECT",
            "statistic":"DGE_STAT",
            "p.value":"P_VALUE"
        }
    )

    out["FDR"] = np.nan

    return out[
        [
            "SYMBOL",
            "assay",
            "DGE_EFFECT",
            "DGE_STAT",
            "P_VALUE",
            "FDR",
            "n.studies",
            "method"
        ]
    ]

In [20]:
dge_3q29_clean = clean_limma_dge(dge_3q29)
dge_15q13_clean = clean_limma_dge(dge_15q13)
dge_idiopathic_clean = clean_dreamlet(dge_idiopathic)
dge_22q11_clean = clean_dreamlet(dge_22q11)
dge_nrxn1_clean = clean_dreamlet(dge_nrxn1)

In [3]:
scz_pgc3 = pd.read_csv('~/programs/MAGMA/results/SCZ_PGC3_gene_results_annotated.csv')

In [66]:
scz_pgc3

,GENE,CHR,START,STOP,NSNPS,NPARAM,N,ZSTAT,P,SYMBOL
0,1,19,58848172,58899865,111,14,168175,0.840250,2.003900e-01,A1BG
1,10,8,18213755,18268723,304,22,168175,0.531800,2.974300e-01,NAT2
2,100,20,43238160,43315376,205,38,168175,0.124370,4.505100e-01,ADA
3,1000,18,25520930,25792445,555,43,168175,-0.390780,6.520200e-01,CDH2
4,10000,1,243641535,244049381,561,24,168175,7.269700,1.800800e-13,AKT3
...,...,...,...,...,...,...,...,...,...,...
18444,9991,9,114969995,115130944,432,15,168175,4.356700,6.601600e-06,PTBP3
18445,9992,21,35701323,35753440,156,29,168175,0.611850,2.703200e-01,KCNE2
18446,9993,22,19013795,19144967,472,23,168175,2.210500,1.353400e-02,DGCR2
18447,9994,6,90504619,90594155,245,29,168175,-0.030628,5.122200e-01,CASP8AP2


In [24]:
scz = scz_pgc3[['SYMBOL','ZSTAT','P']].copy()

scz = scz.rename(columns={'ZSTAT':'SCZ_MAGMA_Z', 'P':'SCZ_MAGMA_P'})

In [67]:
scz

,SYMBOL,SCZ_MAGMA_Z,SCZ_MAGMA_P
0,A1BG,0.840250,2.003900e-01
1,NAT2,0.531800,2.974300e-01
2,ADA,0.124370,4.505100e-01
3,CDH2,-0.390780,6.520200e-01
4,AKT3,7.269700,1.800800e-13
...,...,...,...
18444,PTBP3,4.356700,6.601600e-06
18445,KCNE2,0.611850,2.703200e-01
18446,DGCR2,2.210500,1.353400e-02
18447,CASP8AP2,-0.030628,5.122200e-01


In [68]:
dge_idiopathic_clean_merged = dge_idiopathic_clean.merge(scz, on="SYMBOL", how="inner")

In [70]:
dge_idiopathic_clean_merged.groupby("assay").size()

assay
Dorsal Forebrain Neuron FOXG1+EMX1+NEUROG1+      502
Hindbrain Neurons NR2F2+PBX3+LHX1+               778
Mesenchymal-like cells VIM+VCAN+SPARC+         11341
Mixed Neurons FGF12+GRIN2B+CAMK2B+                 8
Proliferative Radial Glia SOX2+HES6+TOP2A+      8588
Radial Glia SOX2+PAX6+FABP7+                    7938
Radial Glia SOX2+VIM+FABP7+                     2785
dtype: int64

In [76]:
## Correlate Meta-Analytic t statistic with MAGMA z stat #

cor_results = []

for assay, df in dge_idiopathic_clean_merged.groupby("assay"):

    # if len(df) < 100:
    #     continue

    rho, p = spearmanr(df["DGE_STAT"], df["SCZ_MAGMA_Z"])

    cor_results.append({"assay": assay, "n_genes": len(df),  "rho": rho, "p": p}    )

cor_results = pd.DataFrame(cor_results)

cor_results["FDR"] = multipletests(cor_results["p"], method="fdr_bh")[1]

cor_results.sort_values("FDR")

,assay,n_genes,rho,p,FDR
0,Dorsal Forebrain Neuron FOXG1+EMX1+NEUROG1+,502,0.009121,0.838465,0.95013
1,Hindbrain Neurons NR2F2+PBX3+LHX1+,778,0.021773,0.544239,0.95013
2,Mesenchymal-like cells VIM+VCAN+SPARC+,11341,-0.000587,0.950130,0.95013
3,Mixed Neurons FGF12+GRIN2B+CAMK2B+,8,-0.238095,0.570156,0.95013
4,Proliferative Radial Glia SOX2+HES6+TOP2A+,8588,-0.002235,0.835952,0.95013
5,Radial Glia SOX2+PAX6+FABP7+,7938,-0.001977,0.860171,0.95013
6,Radial Glia SOX2+VIM+FABP7+,2785,0.004382,0.817193,0.95013


In [77]:
# Check overlap between between DGE test statistic with MAGMA z stat #

for name, df in {
    "idiopathic": dge_idiopathic_clean,
    "15q13": dge_15q13_clean,
    "3q29": dge_3q29_clean
}.items():

    overlap = set(df["SYMBOL"]) & set(scz_pgc3["SYMBOL"])

    print(name, len(overlap), "overlapping genes")

idiopathic 11401 overlapping genes
15q13 11750 overlapping genes
3q29 11910 overlapping genes


In [78]:
dge_idiopathic_clean["SYMBOL"].nunique(), scz_pgc3["SYMBOL"].nunique()

(13016, 18323)

In [44]:
len(set(dge_idiopathic_clean["SYMBOL"]) - set(scz_pgc3["SYMBOL"]))

1615

In [86]:
OUTDIR = Path("/mnt/sdb/scz_meta_analysis_processed/psychad_mapping/gwas_files")

for celltype in dge_idiopathic_clean["assay"].unique():

    feature = (
        dge_idiopathic_clean
        .loc[dge_idiopathic_clean["assay"] == celltype,
             ["SYMBOL", "DGE_STAT"]]
        .drop_duplicates(subset="SYMBOL")          # one value per gene
        .merge(
            scz_pgc3[["GENE", "SYMBOL"]],
            on="SYMBOL",
            how="inner"
        )
        [["GENE", "DGE_STAT"]]
        .sort_values("GENE")
    )

    outfile = OUTDIR / f"{celltype}_property_idiopathic.txt"
    feature.to_csv(outfile, sep="\t", index=False)

    print(f"{celltype}: {len(feature)} genes -> {outfile.name}")

Mesenchymal-like cells VIM+VCAN+SPARC+: 11341 genes -> Mesenchymal-like cells VIM+VCAN+SPARC+_property_idiopathic.txt
Proliferative Radial Glia SOX2+HES6+TOP2A+: 8588 genes -> Proliferative Radial Glia SOX2+HES6+TOP2A+_property_idiopathic.txt
Radial Glia SOX2+PAX6+FABP7+: 7938 genes -> Radial Glia SOX2+PAX6+FABP7+_property_idiopathic.txt
Radial Glia SOX2+VIM+FABP7+: 2785 genes -> Radial Glia SOX2+VIM+FABP7+_property_idiopathic.txt
Dorsal Forebrain Neuron FOXG1+EMX1+NEUROG1+: 502 genes -> Dorsal Forebrain Neuron FOXG1+EMX1+NEUROG1+_property_idiopathic.txt
Hindbrain Neurons NR2F2+PBX3+LHX1+: 778 genes -> Hindbrain Neurons NR2F2+PBX3+LHX1+_property_idiopathic.txt
Mixed Neurons FGF12+GRIN2B+CAMK2B+: 8 genes -> Mixed Neurons FGF12+GRIN2B+CAMK2B+_property_idiopathic.txt


In [88]:
dge_idiopathic_clean["assay"].unique()

array(['Mesenchymal-like cells VIM+VCAN+SPARC+',
       'Proliferative Radial Glia SOX2+HES6+TOP2A+',
       'Radial Glia SOX2+PAX6+FABP7+', 'Radial Glia SOX2+VIM+FABP7+',
       'Dorsal Forebrain Neuron FOXG1+EMX1+NEUROG1+',
       'Hindbrain Neurons NR2F2+PBX3+LHX1+',
       'Mixed Neurons FGF12+GRIN2B+CAMK2B+'], dtype=object)

In [89]:
import subprocess

MAGMA = "/home/deepak/programs/MAGMA/magma"
GENE_RESULTS = "/home/deepak/programs/MAGMA/results/SCZ_PGC3_gene_results.genes.raw"
OUTDIR = Path("/home/deepak/programs/MAGMA/results")

celltypes = ['Mesenchymal-like cells VIM+VCAN+SPARC+', 'Proliferative Radial Glia SOX2+HES6+TOP2A+',
'Radial Glia SOX2+PAX6+FABP7+', 'Radial Glia SOX2+VIM+FABP7+',
'Dorsal Forebrain Neuron FOXG1+EMX1+NEUROG1+',  'Hindbrain Neurons NR2F2+PBX3+LHX1+']

for celltype in celltypes:

    infile = f"/mnt/sdb/scz_meta_analysis_processed/psychad_mapping/gwas_files/{celltype}_property_idiopathic.txt"
    outfile = OUTDIR / f"{celltype}_property_idiopathic"

    cmd = [
        MAGMA,
        "--gene-results", GENE_RESULTS,
        "--gene-covar", infile,
        "--out", str(outfile)
    ]

    subprocess.run(cmd, check=True)

Welcome to MAGMA v1.10 (linux)
Using flags:
	--gene-results /home/deepak/programs/MAGMA/results/SCZ_PGC3_gene_results.genes.raw
	--gene-covar /mnt/sdb/scz_meta_analysis_processed/psychad_mapping/gwas_files/Mesenchymal-like cells VIM+VCAN+SPARC+_property_idiopathic.txt
	--out /home/deepak/programs/MAGMA/results/Mesenchymal-like cells VIM+VCAN+SPARC+_property_idiopathic

Start time is 01:10:04, Wednesday 08 Jul 2026

Reading file /home/deepak/programs/MAGMA/results/SCZ_PGC3_gene_results.genes.raw... 
	18449 genes read from file
Loading gene-level covariates...
Reading file /mnt/sdb/scz_meta_analysis_processed/psychad_mapping/gwas_files/Mesenchymal-like cells VIM+VCAN+SPARC+_property_idiopathic.txt... 
	detected 1 variables in file (using all)
	found 1 valid gene covariate, for 11341 genes defined in genotype data
Processing missing values...
	found 7108 genes not present in all input files: removing these from analysis
	11341 genes remaining in analysis
Preparing variables for analysis..

In [91]:

RESULTS = Path("/home/deepak/programs/MAGMA/results")

rows = []

for f in RESULTS.glob("*_property_idiopathic.gsa.out"):

    # skip comment lines beginning with '#'
    df = pd.read_csv(f, delim_whitespace=True, comment="#")

    rows.append({
        "assay": f.stem.replace("_property_idiopathic.gsa",""),
        "NGENES": df.loc[0, "NGENES"],
        "BETA": df.loc[0, "BETA"],
        "BETA_STD": df.loc[0, "BETA_STD"],
        "SE": df.loc[0, "SE"],
        "P": df.loc[0, "P"],
    })

results = pd.DataFrame(rows)

results["FDR"] = multipletests(results["P"], method="fdr_bh")[1]

results = results.sort_values("P")

results

,assay,NGENES,BETA,BETA_STD,SE,P,FDR
5,RG_PAX6_DGE,7938,-0.008523,-0.018402,0.006531,0.19191,0.482837
6,Radial Glia SOX2+PAX6+FABP7+,7938,-0.008523,-0.018402,0.006531,0.19191,0.482837
0,Hindbrain Neurons NR2F2+PBX3+LHX1+,778,0.040059,0.069335,0.031714,0.20693,0.482837
2,Mesenchymal-like cells VIM+VCAN+SPARC+,11341,-0.001452,-0.003356,0.005027,0.77271,0.955350
4,Radial Glia SOX2+VIM+FABP7+,2785,-0.002256,-0.002849,0.020720,0.91331,0.955350
1,Dorsal Forebrain Neuron FOXG1+EMX1+NEUROG1+,502,0.004202,0.006404,0.043877,0.92375,0.955350
3,Proliferative Radial Glia SOX2+HES6+TOP2A+,8588,0.000366,0.000734,0.006532,0.95535,0.955350


In [83]:
# # deepak@titan:~/repositories/scz_meta_analysis/psychad_mapping/gwas_analyses$ magma --gene-results /home/deepak/programs/MAGMA/results/SCZ_PGC3_gene_results.genes.raw   --gene-covar /mnt/sdb/scz_meta_analysis_processed/psychad_mapping/gwas_files/RG_PAX6_DGE_property_idiopathic.txt   --out /home/deepak/programs/MAGMA/results/RG_PAX6_DGE_property_idiopathic


# deepak@titan:~/repositories/scz_meta_analysis/psychad_mapping/gwas_analyses$ cat /home/deepak/programs/MAGMA/results/RG_PAX6_DGE_property_idiopathic.gsa.out

# # MEAN_SAMPLE_SIZE = 168175
# # TOTAL_GENES = 7938
# # TEST_DIRECTION = one-sided, positive (set), two-sided (covar)
# # CONDITIONED_INTERNAL = gene size, gene density, inverse mac, log(gene size), log(gene density), log(inverse mac)
# VARIABLE      TYPE  NGENES         BETA     BETA_STD           SE            P
# DGE_STAT     COVAR    7938   -0.0085233    -0.018402    0.0065309      0.19191

# not significant 

In [62]:
## Top 5% MAGMA hits against top 5% of my DGEs by celltype

# background universe
background = set(scz_pgc3["SYMBOL"])

# define thresholds
top_dge = set(
    dge_idiopathic_clean
    .query("DGE_STAT > 1.96")["SYMBOL"]
)

top_scz = set(
    scz_pgc3
    .query("P < 0.05")["SYMBOL"]
)

a = len(top_dge & top_scz)
b = len(top_dge - top_scz)
c = len(top_scz - top_dge)
d = len(background - top_dge - top_scz)

table = [[a,b],
         [c,d]]

oddsratio, p = fisher_exact(table)

print(a, oddsratio, p)

1262 0.9892025926059577 0.7870897055337055


In [93]:
## Top 5% MAGMA hits against top 5% of my DGEs by celltype

# background universe
background = set(scz_pgc3["SYMBOL"])

# define thresholds
top_dge = set(
    dge_idiopathic_clean
    .query("DGE_STAT > 1.96")["SYMBOL"]
)

top_dge = set(dge_idiopathic_clean["SYMBOL"])


top_scz = set(
    scz_pgc3
    .query("P < 0.05")["SYMBOL"]
)

a = len(top_dge & top_scz)
b = len(top_dge - top_scz)
c = len(top_scz - top_dge)
d = len(background - top_dge - top_scz)

table = [[a,b],
         [c,d]]

oddsratio, p = fisher_exact(table)

print(a, oddsratio, p)

4391 1.2983392344853215 7.058184325287785e-16


In [95]:
dge_idiopathic_clean

,SYMBOL,assay,DGE_EFFECT,DGE_STAT,P_VALUE,FDR,n.studies,method
0,A1BG,Mesenchymal-like cells VIM+VCAN+SPARC+,0.355352,0.621290,0.534409,NaN,1,FE
1,A1BG,Proliferative Radial Glia SOX2+HES6+TOP2A+,1.353854,2.133534,0.032881,NaN,1,FE
2,A1BG,Radial Glia SOX2+PAX6+FABP7+,0.309485,0.972774,0.330666,NaN,1,FE
3,A1BG-AS1,Mesenchymal-like cells VIM+VCAN+SPARC+,-0.306088,-0.920000,0.357573,NaN,1,FE
4,AAAS,Mesenchymal-like cells VIM+VCAN+SPARC+,0.109481,0.477147,0.633257,NaN,1,FE
...,...,...,...,...,...,...,...,...
35225,ZZEF1,Mesenchymal-like cells VIM+VCAN+SPARC+,0.216788,1.192933,0.232896,NaN,1,FE
35226,ZZZ3,Mesenchymal-like cells VIM+VCAN+SPARC+,-0.380565,-2.155013,0.031161,NaN,1,FE
35227,ZZZ3,Proliferative Radial Glia SOX2+HES6+TOP2A+,0.090411,0.305267,0.760163,NaN,1,FE
35228,ZZZ3,Radial Glia SOX2+PAX6+FABP7+,-0.257875,-0.941627,0.346384,NaN,1,FE
